In [15]:
# nastavení projektové cesty pro import sdílených modulů

import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))

# knihovny pro práci s daty a HTTP požadavky
import pandas as pd
import requests

# knihovny pro regulární výrazy a práci s časem
import re
import time

# převod RTF souborů do textové podoby
from striprtf.striprtf import rtf_to_text

# sdílené konstanty projektu
from utils.constants import HEADERS
from utils.schema import FINAL_COLUMNS
from utils.text_cleaning import normalize_rtf_text

## Příprava vstupního seznamu předmětů

Byl vytvořen vstupní CSV soubor obsahující kódy relevantních předmětů a URL odkazy na jejich sylaby v systému InSIS. Kódy předmětů byly manuálně převzaty ze studijních plánů vybraných studijních programů.

Do datové sady byly zařazeny pouze povinné předměty. Vynechány byly jazykové předměty a předměty administrativního charakteru, například bakalářské semináře.

Pro získávání dat bylo využito testovací prostředí informačního systému InSIS dostupné na adrese `https://insis-test.vse.cz/`, a nikoliv produkční verze systému.

In [4]:
# načtení vstupního seznamu předmětů
df_courses = pd.read_csv(
    "../data/raw/predmety_vse.csv",
    sep=";",
    header=None,
    usecols=[0, 1],
    names=[
        "course_code",
        "source_url"
    ]
)

# extrakce interního identifikátoru předmětu z URL adresy sylabu
df_courses["predmet_id"] = (
    df_courses["source_url"]
    .str.extract(r"predmet=(\d+)")
)

# kontrola načtených dat
df_courses.head()

,course_code,source_url,predmet_id
0,2OP811,https://insis-test.vse.cz/katalog/syllabus.pl?...,71103
1,2OP816,https://insis-test.vse.cz/katalog/syllabus.pl?...,122664
2,2OP834,https://insis-test.vse.cz/katalog/syllabus.pl?...,71106
3,2OP813,https://insis-test.vse.cz/katalog/syllabus.pl?...,62495
4,2OP833,https://insis-test.vse.cz/katalog/syllabus.pl?...,70956


In [5]:
# kontrola chybějících a duplicitních hodnot

print("Počet NULL hodnot:")
print(df_courses.isnull().sum())

print("\nPočet unikátních předmětů:")
print(df_courses["course_code"].nunique())

print("\nPočet unikátních interních ID:")
print(df_courses["predmet_id"].nunique())

Počet NULL hodnot:
course_code    0
source_url     0
predmet_id     0
dtype: int64

Počet unikátních předmětů:
133

Počet unikátních interních ID:
133


In [6]:
# stažení RTF sylabů z testovacího prostředí InSIS

INSIS_URL = "https://insis-test.vse.cz/katalog/syllabus.pl"

RTF_DIR = Path("../data/raw/vse_rtf")
RTF_DIR.mkdir(parents=True, exist_ok=True)


def download_rtf(predmet_id, course_code, out_dir=RTF_DIR):

    payload = {
        "vystup": 3,
        "zobrazit_vystup": "Zobrazit",
        "predmet": str(predmet_id),
        "typ": 1,
        "jazyk": 3
    }

    response = requests.post(
        INSIS_URL,
        data=payload,
        headers=HEADERS,
        timeout=15
    )

    response.raise_for_status()

    file_path = out_dir / f"{course_code}.rtf"

    file_path.write_bytes(response.content)

    return file_path

In [9]:
# stažení všech RTF sylabů

downloaded_files = []
failed_downloads = []

for _, row in df_courses.iterrows():

    print(f"Stahuji {row['course_code']}")

    try:
        file_path = download_rtf(
            predmet_id=row["predmet_id"],
            course_code=row["course_code"]
        )

        downloaded_files.append(file_path)

    except Exception as e:

        failed_downloads.append({
            "course_code": row["course_code"],
            "predmet_id": row["predmet_id"],
            "error": str(e)
        })

        print(f"CHYBA: {e}")

    time.sleep(2)

print(f"\nStaženo souborů: {len(downloaded_files)}")
print(f"Neúspěšná stažení: {len(failed_downloads)}")

Stahuji 2OP811
Stahuji 2OP816
Stahuji 2OP834
Stahuji 2OP813
Stahuji 2OP833
Stahuji 2OP853
Stahuji 2OP863
Stahuji 2OP823
Stahuji 2OP841
Stahuji 2OP831
Stahuji 2OP827
Stahuji 2OP857
Stahuji 2OP822
Stahuji 2OP842
Stahuji 2OP862
Stahuji 2OP832
Stahuji 2OP844
Stahuji 2OP821
Stahuji 2OP851
Stahuji 2OP843
Stahuji 2OP815
Stahuji 2OP835
Stahuji 2OP852
Stahuji 2OP812
Stahuji 3BE226
Stahuji 3BE122
Stahuji 3BE114
Stahuji 3BE315
Stahuji 3BE113
Stahuji 3BE116
Stahuji 3BE225
Stahuji 3BE324
Stahuji 3BE322
Stahuji 3BE316
Stahuji 3BE223
Stahuji 3BE112
Stahuji 3BE123
Stahuji 3BE323
Stahuji 3PS124
Stahuji 3BE214
Stahuji 3BE216
Stahuji 3BE312
Stahuji 4ST601
Stahuji 3SG205
Stahuji 2PR152
Stahuji 3BE314
Stahuji 3PO633
Stahuji 4IT534
Stahuji 4IT419
Stahuji 4IT529
Stahuji 4IT526
Stahuji 4IT535
Stahuji 4IT533
Stahuji 4IT553
Stahuji 4IT543
Stahuji 4IT537
Stahuji 5IE756
Stahuji 1BP260
Stahuji 5EN773
Stahuji 5EN757
Stahuji 5EN755
Stahuji 5IE752
Stahuji 5EN775
Stahuji 5EN759
Stahuji 5ZP720
Stahuji 5HP700
Stahuji 5H

In [10]:
# načtení RTF souboru a převod do textové podoby

def read_rtf_file(file_path):

    with open(
        file_path,
        "r",
        encoding="utf-8",
        errors="ignore"
    ) as file:

        rtf_content = file.read()

    text_content = rtf_to_text(rtf_content)

    return text_content

In [11]:
# definice názvů sekcí extrahovaných ze sylabu

FIELDS = {
    "course_code": "Course code:",
    "course_title": "Course title in English:",
    "credits": "Number of ECTS credits allocated:",
    "semester": "Semester:",
    "study_level": "Level of course and year of study:",
    "aims": "Aims of the course:",
    "learning_outcomes": "Learning outcomes and competences:",
    "course_content": "Course contents:",
    "assessment_methods": "Assessment methods and criteria:",
    "literature": "Reading:"
}

In [13]:
# metadata akademického období byla převzata ze studijních plánů, ze kterých byly sylaby získávány
# většina programů odpovídá období WS 2026/2027
# dva programy však neměly dostupné sylaby pro období 2026/2027, a proto bylo použito období WS 2025/2026

DEFAULT_ACADEMIC_YEAR = "2026/2027"
DEFAULT_SEMESTER = "WS 2026/2027"

SPECIAL_PERIODS = {
    "1212": {
        "academic_year": "2025/2026",
        "semester": "WS 2025/2026"
    },
    "1264": {
        "academic_year": "2025/2026",
        "semester": "WS 2025/2026"
    }
}

In [14]:
# určení akademického období podle parametru poc_obdobi

def get_period_metadata(source_url: str) -> dict:

    match = re.search(
        r"poc_obdobi=(\d+)",
        str(source_url)
    )

    if match:

        poc_obdobi = match.group(1)

        if poc_obdobi in SPECIAL_PERIODS:
            return SPECIAL_PERIODS[poc_obdobi]

    return {
        "academic_year": DEFAULT_ACADEMIC_YEAR,
        "semester": DEFAULT_SEMESTER
    }

In [16]:
# funkce pro extrakci krátkých a delších sekcí ze sylabu

def extract_single_line(text: str, label: str) -> str:
    """
    Vrátí první neprázdný řádek za zadaným štítkem.
    """

    start = text.find(label)

    if start == -1:
        return ""

    start = start + len(label)

    rest = text[start:].splitlines()

    for line in rest:
        line = normalize_rtf_text(line)

        if line:
            return line

    return ""


def extract_between(text: str, start_label: str, all_labels: list[str]) -> str:
    """
    Vrátí text mezi zadaným štítkem a následujícím známým štítkem.
    """

    start = text.find(start_label)

    if start == -1:
        return ""

    start = start + len(start_label)

    end_positions = []

    for label in all_labels:

        if label == start_label:
            continue

        position = text.find(label, start)

        if position != -1:
            end_positions.append(position)

    end = min(end_positions) if end_positions else len(text)

    return normalize_rtf_text(text[start:end])


def remove_after_markers(text: str, markers: list[str]) -> str:
    """
    Odstraní část textu za prvním nalezeným ukončovacím markerem.
    """

    if not text:
        return ""

    positions = [
        text.find(marker)
        for marker in markers
        if text.find(marker) != -1
    ]

    if positions:
        text = text[:min(positions)]

    return normalize_rtf_text(text)

In [17]:
# pomocné funkce pro normalizaci hodnot a úpravu sekcí sylabu

def extract_first_number(value: str) -> str:
    """
    Vrátí první číslo nalezené v textu.
    """

    match = re.search(r"\d+", str(value))

    return match.group(0) if match else ""


def normalize_semester(value: str) -> str:
    """
    Převede označení semestru na winter/summer.
    """

    value = str(value).lower()

    if "ws" in value:
        return "winter"

    if "ss" in value:
        return "summer"

    return ""


def clean_assessment(text: str) -> str:
    """
    Ponechá pouze jednotlivé položky hodnocení s procentuálním zastoupením.
    """

    if not text:
        return ""

    # odstranění obecné hodnoticí škály a navazujících sekcí
    cut_markers = [
        "Assessment:",
        "Graded courses",
        "Ungraded courses",
        "Special requirements and details:"
    ]

    text = remove_after_markers(text, cut_markers)

    lines = text.splitlines()
    cleaned = []

    for line in lines:

        line = normalize_rtf_text(line)

        match = re.match(r"(.+?)(\d+\s*%)$", line)

        if match:

            name = match.group(1).strip()
            percent = match.group(2).replace(" ", "")

            if name.lower() != "total":
                cleaned.append(f"{name}: {percent}")

    return "\n".join(cleaned)

In [18]:
# převod RTF sylabu do strukturovaného záznamu

def parse_syllabus(path: Path, source_url: str) -> dict:

    raw_text = normalize_rtf_text(
        read_rtf_file(path)
    )

    labels = list(FIELDS.values())

    period_metadata = get_period_metadata(source_url)

    row = {
        "university": "Prague University of Economics and Business",
        "country": "Czech Republic",
        "source_url": source_url,
        "academic_year": period_metadata["academic_year"],
        "semester": period_metadata["semester"],
        "semester_normalized": normalize_semester(period_metadata["semester"]),
        "study_level": "",
        "credits": "",
        "credit_system": "ECTS",
        "ects_credits": "",
        "course_title": "",
        "course_code": "",
        "aims": "",
        "course_content": "",
        "learning_outcomes": "",
        "assessment_methods": "",
        "literature": ""
    }

    short_fields = [
        "course_code",
        "course_title",
        "credits",
        "study_level"
    ]

    long_fields = [
        "aims",
        "learning_outcomes",
        "course_content",
        "assessment_methods",
        "literature"
    ]

    for field in short_fields:
        row[field] = extract_single_line(
            raw_text,
            FIELDS[field]
        )

    for field in long_fields:
        row[field] = extract_between(
            raw_text,
            FIELDS[field],
            labels
        )

    # odstranění části, která v exportu InSIS navazuje na obsah předmětu,
    # ale patří k pracovní zátěži a nikoliv k tematickému obsahu
    row["course_content"] = remove_after_markers(
        row["course_content"],
        ["Learning activities, teaching methods and workload"]
    )

    # extrakce položek hodnocení s procentuálním zastoupením
    row["assessment_methods"] = clean_assessment(
        row["assessment_methods"]
    )

    row["credits"] = extract_first_number(
        row["credits"]
    )

    row["ects_credits"] = row["credits"]

    study_level = row["study_level"].lower()

    if "bachelor" in study_level:
        row["study_level"] = "bachelor"

    elif "master" in study_level:
        row["study_level"] = "master"

    else:
        row["study_level"] = ""

    return row

In [21]:
# převod všech RTF sylabů do strukturovaného datasetu

parsed_rows = []
parsing_errors = []

for _, row in df_courses.iterrows():

    file_path = RTF_DIR / f"{row['course_code']}.rtf"

    if not file_path.exists():

        print(f"Soubor chybí: {file_path}")

        parsing_errors.append({
            "course_code": row["course_code"],
            "error": "missing file"
        })

        continue

    try:

        parsed_row = parse_syllabus(
            path=file_path,
            source_url=row["source_url"]
        )

        parsed_rows.append(parsed_row)

    except Exception as e:

        parsing_errors.append({
            "course_code": row["course_code"],
            "error": str(e)
        })

        print(f"CHYBA při parsování {row['course_code']}: {e}")

df_vse = pd.DataFrame(parsed_rows)
df_vse = df_vse[FINAL_COLUMNS]

print(f"\nÚspěšně naparsovaných sylabů: {len(df_vse)}")
print(f"Chyby při parsování: {len(parsing_errors)}")


Úspěšně naparsovaných sylabů: 133
Chyby při parsování: 0


In [24]:
# uložení naparsovaného interim datasetu

INTERIM_OUTPUT_PATH = "../data/interim/vse_syllabi_raw_parsed.csv"

df_vse.to_csv(
    INTERIM_OUTPUT_PATH,
    index=False,
    encoding="utf-8-sig"
)

print(f"Soubor byl uložen do: {INTERIM_OUTPUT_PATH}")

Soubor byl uložen do: ../data/interim/vse_syllabi_raw_parsed.csv
